# ADAPTIVE RAG Pipeline
### PDF → Chunking → FAISS → Query Classifier → Router (Simple / Medium / Complex) → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")
print(f"Sample Chunk:\n{chunks[0].page_content}")

Total Chunks: 136
Sample Chunk:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director General of Defence Research and
Development Organisation
In office
1992–1999
A. P. J. Abdul Kalam
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl
kəˈlɑːm/ ⓘ UB-duul k ə -LAHM; 15 October 1931 –
27 July 2015) was an Indian aerospace scientist and
statesman who served as the president of India from
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a


## Step 3: Create Embeddings & Store in FAISS

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Setup LLM

In [5]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

## Step 5: Query Classifier — Analyze query complexity

In [6]:
def classify_query(query):
    prompt = f"""Classify the complexity of this query into exactly one category.

- "simple" → single fact lookup (e.g., birth year, name, title)
- "medium" → needs combining 2-3 facts or slight reasoning
- "complex" → needs multi-step reasoning, comparison, or synthesis across topics

Query: {query}
Respond with only one word: simple, medium, or complex."""

    response = llm.invoke(prompt).content.strip().lower()
    for level in ["simple", "medium", "complex"]:
        if level in response:
            return level
    return "simple"

## Step 6: Simple Pipeline — Top-3 retrieval

In [7]:
def simple_pipeline(query):
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

    answer = llm.invoke(prompt).content
    return answer, docs

## Step 7: Medium Pipeline — Multi-query retrieval

In [8]:
def medium_pipeline(query):
    # Generate multiple search queries for broader retrieval
    prompt = f"""Generate 3 different search queries to help answer this question.
Each query should target a different aspect of the question.

Question: {query}

Return only the 3 queries, one per line, no numbering."""

    response = llm.invoke(prompt).content.strip()
    sub_queries = [q.strip() for q in response.split("\n") if q.strip()][:3]
    print(f"  Sub-queries: {sub_queries}")

    # Retrieve docs for each sub-query and merge (deduplicate)
    all_docs = []
    seen = set()
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    for sq in sub_queries:
        docs = retriever.invoke(sq)
        for doc in docs:
            doc_id = doc.page_content[:100]
            if doc_id not in seen:
                seen.add(doc_id)
                all_docs.append(doc)

    print(f"  Merged unique chunks: {len(all_docs)}")

    context = "\n\n".join([doc.page_content for doc in all_docs])
    answer_prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

    answer = llm.invoke(answer_prompt).content
    return answer, all_docs

## Step 8: Complex Pipeline — Multi-step retrieval

In [9]:
def complex_pipeline(query):
    # Break the complex query into sequential steps
    prompt = f"""Break this complex question into 2-3 simpler sub-questions that should be answered in order.
Each sub-question should build on the previous one.

Question: {query}

Return only the sub-questions, one per line, no numbering."""

    response = llm.invoke(prompt).content.strip()
    steps = [s.strip() for s in response.split("\n") if s.strip()][:3]
    print(f"  Steps: {steps}")

    # Answer each step sequentially, building context
    all_docs = []
    accumulated_answers = []
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    for i, step in enumerate(steps):
        docs = retriever.invoke(step)
        all_docs.extend(docs)
        context = "\n\n".join([doc.page_content for doc in docs])

        step_prompt = f"""Answer the question based only on the following context:

{context}

{"Previous findings: " + " | ".join(accumulated_answers) if accumulated_answers else ""}

Question: {step}
Answer:"""

        step_answer = llm.invoke(step_prompt).content
        accumulated_answers.append(step_answer.strip())
        print(f"  Step {i+1}: {step} → {step_answer.strip()[:100]}...")

    # Final synthesis
    final_prompt = f"""Based on these findings, provide a complete answer to the original question.

Findings:
{chr(10).join(f"- {a}" for a in accumulated_answers)}

Original Question: {query}
Answer:"""

    answer = llm.invoke(final_prompt).content
    return answer, all_docs

## Step 9: Router — Select strategy based on classification

In [10]:
def route_query(query, complexity):
    if complexity == "simple":
        return simple_pipeline(query)
    elif complexity == "medium":
        return medium_pipeline(query)
    else:
        return complex_pipeline(query)

## Step 10: Full Adaptive RAG Pipeline

In [11]:
def adaptive_rag(query):
    print(f"Query: {query}\n")

    # Classify query complexity
    complexity = classify_query(query)
    print(f"[CLASSIFIER] Complexity: {complexity}")

    # Route to the right pipeline
    print(f"[ROUTER] Using {complexity} pipeline\n")
    answer, docs = route_query(query, complexity)

    print(f"\nAnswer: {answer}")

    # Show source chunks
    print(f"\n--- Retrieved {len(docs)} chunks ---")
    for i, doc in enumerate(docs):
        page = doc.metadata.get("page", "?")
        print(f"  Chunk {i+1} (Page {int(page)+1}): {doc.page_content[:100]}...")

    return answer

## Run Adaptive RAG — Simple Query

In [12]:
answer = adaptive_rag("Where was Abdul Kalam born?")

Query: Where was Abdul Kalam born?

[CLASSIFIER] Complexity: simple
[ROUTER] Using simple pipeline


Answer: Abdul Kalam was born in Rameswaram, Tamil Nadu, on Pamban Island.

--- Retrieved 3 chunks ---
  Chunk 1 (Page 2): Organisation
Website A. P. J. Abdul Kalam Centre (h
ttps://kalamcentre.com)
Signature
Kalam's birthp...
  Chunk 2 (Page 10): of Tamil Nadu announced that Kalam's birthday, 15 October, would be observed as "Youth Renaissance
D...
  Chunk 3 (Page 11): A P J Abdul Kalam: The Visionary of India by K Bhushan, G Katyal; A P H Pub Corp,
2002.[176]
The Kal...


## Run Adaptive RAG — Medium Query

In [13]:
answer = adaptive_rag("What did Kalam study and where did he work after that?")

Query: What did Kalam study and where did he work after that?

[CLASSIFIER] Complexity: medium
[ROUTER] Using medium pipeline

  Sub-queries: ['What were the educational background and career paths of the Indian President Abdul Kalam?', 'What were the educational background and career paths of the Indian President Abdul Kalam?', 'What were the educational background and career paths of the Indian President Abdul Kalam?']
  Merged unique chunks: 3

Answer: Kalam studied physics and aerospace engineering. After that, he worked at the Madras Institute of Technology, where he studied aerospace engineering.

--- Retrieved 3 chunks ---
  Chunk 1 (Page 3): This was my first stage, in which I learnt
leadership from three great teachers—Dr
Vikram Sarabhai, ...
  Chunk 2 (Page 1): 2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics an...
  Chunk 3 (Page 2): Sri Lanka who claim descent from Arab traders and
local women. The family business had involve

## Run Adaptive RAG — Complex Query

In [14]:
answer = adaptive_rag("How did Kalam's early education influence his role in India's missile program and later presidency?")

Query: How did Kalam's early education influence his role in India's missile program and later presidency?

[CLASSIFIER] Complexity: complex
[ROUTER] Using complex pipeline

  Steps: ["What was Kalam's early education like, and how did it shape his academic and professional pursuits?", "How did Kalam's academic achievements and interests in science and technology influence his decision to pursue a career in engineering and research?", "How did Kalam's experiences and skills gained during his time in the Indian Air Force and as a researcher at the ISRO influence his role in India's missile program and later presidency?"]
  Step 1: What was Kalam's early education like, and how did it shape his academic and professional pursuits? → According to the text, Kalam's early education was shaped by his learning from three great teachers:...
  Step 2: How did Kalam's academic achievements and interests in science and technology influence his decision to pursue a career in engineering and researc

Test Questions

1. What year was Abdul Kalam born?
2. What was the name of the coronary stent Kalam co-developed?
3. Which institution did Kalam study aerospace engineering at?
4. How many electoral votes did Kalam receive in the 2002 presidential election?
5. What was Kalam's role at DRDO from 1992 to 1999?